# Akosile Fast Transcriber

This notebook transcribes a recording with Whisper Large v3 Turbo on a free Google graphics card. An hour of audio usually takes a few minutes.

**How to use it**

1. Open the **Runtime** menu, choose **Change runtime type**, select **T4 GPU** and click **Save**.
2. Fill in the settings in step 1 below.
3. Open the **Runtime** menu and choose **Run all**.
4. When step 3 shows a **Choose files** button, pick your recording.
5. Your browser downloads two files when the notebook finishes. Open the `.srt` file in Akosile with **Import a transcript**. The `.zip` file holds the same transcript as SRT, VTT, TXT and JSON.

Your audio stays on this temporary Google machine, which Google deletes when the session ends.

In [ ]:
#@title 1. Settings
#@markdown Choose the language the speakers use. Setting it yourself stops Whisper from guessing wrong on a Nigerian accent.
LANGUAGE = "English" #@param ["English", "Yoruba", "Hausa", "Swahili", "French", "Detect it for me"]
#@markdown Choose **Translate to English** to get an English transcript of a recording in another language.
OUTPUT = "Same language" #@param ["Same language", "Translate to English"]
#@markdown List names and terms the speakers use, separated by commas. Whisper spells these correctly more often when you list them.
NAMES = "Lagos, Ikeja, naira" #@param {type:"string"}
#@markdown Leave this empty to upload a file in step 3. To use a file in your Google Drive instead, paste its path, for example `/content/drive/MyDrive/Recordings/call.m4a`.
DRIVE_PATH = "" #@param {type:"string"}

LANGUAGE_CODES = {"English": "en", "Yoruba": "yo", "Hausa": "ha", "Swahili": "sw", "French": "fr", "Detect it for me": None}
LANG = LANGUAGE_CODES[LANGUAGE]
TASK = "translate" if OUTPUT == "Translate to English" else "transcribe"
# Large v3 Turbo was not trained to translate, so translation uses the full Large v3 model.
MODEL = "large-v3" if TASK == "translate" else "large-v3-turbo"
print(f"Akosile will use Whisper {MODEL} for {LANGUAGE.lower()} audio.")

In [ ]:
#@title 2. Prepare the graphics card and install Whisper
import subprocess, sys
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu.returncode != 0:
    print("This session has no graphics card. Open Runtime, choose Change runtime type, select T4 GPU, then run all cells again.")
    print("The notebook will still work on the processor, but an hour of audio will take much longer.")
    DEVICE = "cpu"
else:
    print("Graphics card ready:", gpu.stdout.strip())
    DEVICE = "cuda"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper>=1.1"], check=True)
print("Whisper is installed.")

In [ ]:
#@title 3. Choose your recording
import os
if DRIVE_PATH.strip():
    from google.colab import drive
    drive.mount("/content/drive")
    AUDIO_PATH = DRIVE_PATH.strip()
    if not os.path.exists(AUDIO_PATH):
        raise FileNotFoundError(f"Google Drive has no file at {AUDIO_PATH}. Check the path in step 1.")
else:
    from google.colab import files
    print("Click Choose files and pick your recording. A large file can take a few minutes to upload.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded. Run this cell again and choose a recording.")
    AUDIO_PATH = os.path.abspath(next(iter(uploaded)))
print("Recording:", os.path.basename(AUDIO_PATH))

In [ ]:
#@title 4. Transcribe
import glob, os, time, ctypes, subprocess, sys
from faster_whisper import WhisperModel, BatchedInferencePipeline

def load_cuda_libraries():
    """Installs and preloads the CUDA libraries Whisper needs if Colab's own copies do not match."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"], check=True)
    import nvidia
    root = list(nvidia.__path__)[0]
    for pattern in ["cublas/lib/libcublas*.so*", "cudnn/lib/libcudnn*.so*"]:
        for lib in sorted(glob.glob(os.path.join(root, pattern))):
            try:
                ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass

def load_model():
    if DEVICE == "cuda":
        try:
            return WhisperModel(MODEL, device="cuda", compute_type="float16")
        except Exception as err:
            if "cudnn" in str(err).lower() or "cublas" in str(err).lower():
                print("Fixing the graphics card libraries. This takes about a minute.")
                load_cuda_libraries()
                return WhisperModel(MODEL, device="cuda", compute_type="float16")
            raise
    return WhisperModel(MODEL, device="cpu", compute_type="int8")

print(f"Loading Whisper {MODEL}. The first load downloads about 1.5 GB.")
model = load_model()
pipeline = BatchedInferencePipeline(model=model)

names = ", ".join(n.strip() for n in NAMES.split(",") if n.strip())
started = time.time()
segments_iter, info = pipeline.transcribe(
    AUDIO_PATH,
    language=LANG,
    task=TASK,
    batch_size=16 if DEVICE == "cuda" else 4,
    vad_filter=True,
    without_timestamps=False,
    hotwords=names or None,
)
total = info.duration or 0
print(f"The recording is {int(total // 3600)}h {int(total % 3600 // 60)}m long. Whisper hears {info.language} speech.")

SEGMENTS = []
next_report = 0.0
for seg in segments_iter:
    text = seg.text.strip()
    if text:
        SEGMENTS.append({"start": round(seg.start, 2), "end": round(seg.end, 2), "text": text})
    if total and seg.end / total >= next_report:
        print(f"  {seg.end / total:4.0%} done, up to {int(seg.end // 60)} min")
        next_report += 0.1

elapsed = time.time() - started
print(f"Whisper finished {len(SEGMENTS)} lines in {elapsed / 60:.1f} minutes.")

In [ ]:
#@title 5. Save and download the transcript
import json, os, zipfile
from google.colab import files

def stamp(seconds, sep):
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d}{sep}{ms:03d}"

def clock(seconds):
    seconds = int(seconds)
    h, m, s = seconds // 3600, seconds % 3600 // 60, seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

def write_all(segments, base, duration):
    with open(base + ".srt", "w", encoding="utf-8") as f:
        for i, s in enumerate(segments, 1):
            f.write(f"{i}\n{stamp(s['start'], ',')} --> {stamp(s['end'], ',')}\n{s['text']}\n\n")
    with open(base + ".vtt", "w", encoding="utf-8") as f:
        f.write("WEBVTT\n\n")
        for s in segments:
            f.write(f"{stamp(s['start'], '.')} --> {stamp(s['end'], '.')}\n{s['text']}\n\n")
    with open(base + ".txt", "w", encoding="utf-8") as f:
        for s in segments:
            f.write(f"[{clock(s['start'])}] {s['text']}\n")
    with open(base + ".json", "w", encoding="utf-8") as f:
        json.dump({"app": "akosile", "name": os.path.basename(base), "duration": duration, "segments": segments}, f, ensure_ascii=False, indent=1)
    with zipfile.ZipFile(base + ".zip", "w", zipfile.ZIP_DEFLATED) as z:
        for ext in ("srt", "vtt", "txt", "json"):
            z.write(base + "." + ext, os.path.basename(base) + "." + ext)

base = os.path.join("/content", os.path.splitext(os.path.basename(AUDIO_PATH))[0])
write_all(SEGMENTS, base, total)
print("Your browser is downloading the transcript. Open the .srt file in Akosile with Import a transcript.")
files.download(base + ".srt")
files.download(base + ".zip")